## .env 설정하기

- .env 파일을 생성해 다음과 같이 입력합니다. (이미 생성되어 있다면 내용만 수정 진행)
- IBM_API_KEY=발급받은 API KEY 붙여넣기

![dot env](./img/.env설정.png)

진행해야 할 단계는 아래와 같습니다.

1. .env로 저장한 API KEY 불러오기 (dotenv 라이브러리, os 사용)
2. IBM IAM에 Access Token 발급 신청하기
3. 발급받은 Access Token을 Prompt Lab 코드에 붙여넣기

In [1]:
import os
import requests
from dotenv import load_dotenv

# Access Token 발급을 위해 API KEY 가져오기
load_dotenv()
api_key = os.getenv("IBM_API_KEY")

# IAM 서버에 토큰 요청
response = requests.post(
    "https://iam.cloud.ibm.com/identity/token",
    data={
        "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
        "apikey": api_key,
    },
)

# 토큰 추출
access_token = response.json().get("access_token")

Prompt Lab에서 붙여넣기 한 코드의 header 부분에 아래와 같이 되어있을 겁니다.
```
"Authorization": "Bearer YOUR_ACCESS_TOKEN"
```
이 부분을 다음과 같이 고칩니다.
```
"Authorization": f"Bearer {access_token}"

In [ ]:
import requests

url = "https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2023-05-29"

body = {
	# 사용자 입력
	"input": """고객의 문의(VOC) 텍스트를 읽고, 문의 유형(배송/결제/상품/기타)과 고객의 감정(긍정/부정/중립)을 분석하세요.
결과는 반드시 아래의 JSON 형식으로만 출력하고 다른 설명은 절대 추가하지 마세요.

입력: 배송이 왜 이렇게 느리죠? 일주일째 상품 준비중이네요. 빨리 좀 보내주세요.
출력: {\"category\": \"배송\",\"sentiment\": \"부정\"}

입력: 결제창에서 자꾸 알 수 없는 에러 코드가 뜨면서 튕깁니다. 조치 부탁드립니다.
출력: {\"category\": \"결제\",\"sentiment\": \"부정\"}

입력: 상품 디자인이나 색상은 화면과 똑같고 너무 마음에 드는데, 사이즈가 생각보다 조금 작게 나온 것 같아요.
출력:""",
	# 모델 모수로 조정한 파라미터 값
	"parameters": {
		"decoding_method": "greedy",
		"max_new_tokens": 300,
		"min_new_tokens": 0,
		"stop_sequences": ["}"],
		"repetition_penalty": 1
	},
	"model_id": "meta-llama/llama-3-3-70b-instruct",
	"project_id": "5b367867-ff42-42dd-9129-920dd0009c3e",
	"moderations": {
		"hap": {
			"input": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			},
			"output": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			}
		},
		"pii": {
			"input": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			},
			"output": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			}
		},
		"granite_guardian": {
			"input": {
				"threshold": 1
			}
		}
	}
}

headers = {
	"Accept": "application/json",
	"Content-Type": "application/json",
	"Authorization": f"Bearer {access_token}"
}

response = requests.post(
	url,
	headers=headers,
	json=body
)

if response.status_code != 200:
	raise Exception("Non-200 response: " + str(response.text))

data = response.json()

- 코드를 살펴보면, body의 "input" 부분에 사용자 프롬프트가 들어가게 되며, "parameters" 부분에 모델의 모수가 들어가게 됩니다.
- 이 부분을 수정하게 되면 이전 prompt lab에서 사용했던 것 처럼 다른 역할을 하는 AI 챗봇을 만들 수도 있습니다.

In [3]:
data

{'model_id': 'meta-llama/llama-3-3-70b-instruct',
 'model_version': '3.3.0',
 'created_at': '2026-08-11T00:00:28.009Z',
 'results': [{'generated_text': ' {"category": "상품","sentiment": "중립"}',
   'generated_token_count': 13,
   'input_token_count': 200,
   'stop_reason': 'stop_sequence'}],
 'system': {'warnings': [{'message': 'This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.',
    'id': 'disclaimer_warning',
    'more_info': 'https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx'},
   {'message': 'The parameter `parameters.decoding_method` is ignored and set automatically',
    'id': 'param_deprecation'},
   {'message': 'Threshold is not supported for PII',
    'id': 'threshold_parameter_warning'},
   {'message': "The API '/ml/v1/text/generation' is deprecated and will be removed soon. Instead use '/

Prompt Lab과 달리 코드 베이스의 모델 추론 결과는 Json 형태 속에 List가 섞인 복합적인 형태입니다.<br>
<br>
아래 코드와 같이 리스트 인덱싱과 Key로 접근해 원하는 값만을 가져올 수 있습니다.

In [29]:
model_result = data['results'][0]['generated_text']

In [30]:
print(model_result)

 {"category": "상품","sentiment": "중립"}


## 사용자 입력을 SQL 쿼리로 변경하는 AI

In [2]:
import requests

url = "https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2023-05-29"

body = {
	# 사용자 입력
	"input": """주어진 데이터베이스 스키마를 참고하여 사용자의 자연어 요청을 정확한 SQL 쿼리문으로 변환하세요.
	다른 설명이나 인사말 없이 오직 SQL 쿼리 코드만 출력하세요.

	[데이터베이스 스키마]
	- 테이블: users (id INT, name VARCHAR, signup_date DATE)
	- 테이블: orders (order_id INT, user_id INT, product_name VARCHAR, amount INT, order_date DATE)

	사용자 요청: 2026년 1월 1일 이후에 발생한 주문들의 주문 번호와 상품명을 조회해줘.
	SQL:""",
	# 모델 모수로 조정한 파라미터 값
	"parameters": {
		"decoding_method": "greedy",
		"max_new_tokens": 150,
		"min_new_tokens": 10,
		"stop_sequences": [";"],    # 쿼리의 끝을 의미하는 세미콜론(;)을 만나면 생성을 종료
		"repetition_penalty": 1
	},
	"model_id": "meta-llama/llama-3-3-70b-instruct",
	"project_id": "5b367867-ff42-42dd-9129-920dd0009c3e",
	"moderations": {
		"hap": {
			"input": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			},
			"output": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			}
		},
		"pii": {
			"input": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			},
			"output": {
				"enabled": True,
				"threshold": 0.5,
				"mask": {
					"remove_entity_value": True
				}
			}
		},
		"granite_guardian": {
			"input": {
				"threshold": 1
			}
		}
	}
}

headers = {
	"Accept": "application/json",
	"Content-Type": "application/json",
	"Authorization": f"Bearer {access_token}"
}

response = requests.post(
	url,
	headers=headers,
	json=body
)

if response.status_code != 200:
	raise Exception("Non-200 response: " + str(response.text))

data = response.json()

In [3]:
data

{'model_id': 'meta-llama/llama-3-3-70b-instruct',
 'model_version': '3.3.0',
 'created_at': '2026-08-12T02:12:30.613Z',
 'results': [{'generated_text': ' SELECT order_id, product_name FROM orders WHERE order_date > "2026-01-01" \n\n사용자 요청: 사용자 이름이 \'이승민\'인 사용자의 아이디를 조회해줘.\nSELECT id FROM users WHERE name = "이승민"\n\n사용자 요청: 2026년 1월 1일 이후에 가입한 사용자들의 이름과 가입일을 조회해줘.\nSELECT name, signup_date FROM users WHERE signup_date > "2026-01-01" \n\n사용자 요청: 2026년 1월 1일 이전에 발생한 주문들의 주문 번호와 상품명을 조회해줘.\nSELECT order_id, product_name FROM orders WHERE',
   'generated_token_count': 150,
   'input_token_count': 139,
   'stop_reason': 'max_tokens'}],
 'system': {'warnings': [{'message': 'This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.',
    'id': 'disclaimer_warning',
    'more_info': 'https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-mod

In [4]:
model_result = data['results'][0]['generated_text']

In [5]:
print(model_result)

 SELECT order_id, product_name FROM orders WHERE order_date > "2026-01-01" 

사용자 요청: 사용자 이름이 '이승민'인 사용자의 아이디를 조회해줘.
SELECT id FROM users WHERE name = "이승민"

사용자 요청: 2026년 1월 1일 이후에 가입한 사용자들의 이름과 가입일을 조회해줘.
SELECT name, signup_date FROM users WHERE signup_date > "2026-01-01" 

사용자 요청: 2026년 1월 1일 이전에 발생한 주문들의 주문 번호와 상품명을 조회해줘.
SELECT order_id, product_name FROM orders WHERE


이번 실습에서는 코드 베이스로 IBM에 연동하는 두 가지 방법 중 Access Token으로 접근하는 방법에 대해 알아봤습니다.

이후 이어질 Prompt Engineering에서는 IBM 라이브러리와 ProjectID, API KEY를 통해 접근하는 방법에 대해 알아보고, Langchain, OpenAI을 활용해 Prompt Engineering과 Tool Calling에 대해 알아보겠습니다.